# 02. 저장된 raw 데이터 전처리
인터넷/API 호출 없이 반복 실행할 수 있습니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.preprocess import build_ml_dataset
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
final_df_raw = pd.read_parquet(RAW_DIR / 'final_df_raw.parquet')
collection_missing_df = pd.read_csv(RAW_DIR / 'final_missing_df.csv')
sp500_universe = pd.read_csv(RAW_DIR / 'sp500_universe.csv')
print(final_df_raw.shape, collection_missing_df.shape, sp500_universe.shape)

(1284440, 8) (1, 7) (503, 4)


In [3]:
final_df, coverage_df, final_missing_df = build_ml_dataset(
    final_df_raw, universe=sp500_universe, final_missing_df=collection_missing_df
)
display(final_df.head())
display(coverage_df.head())
display(final_missing_df.head())

,Date,Ticker,Open,High,Low,Close,Volume,source
0,2016-01-04,A,37.751921,37.871444,37.089928,37.411728,3287300,yahoo
1,2016-01-05,A,37.448514,37.650791,37.089936,37.283016,2587200,yahoo
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo
4,2016-01-08,A,36.060178,36.510699,35.370603,35.480934,3736700,yahoo


,Ticker,n_rows,actual_start_date,actual_end_date,n_price_imputed,n_ohlc_inconsistent,n_volume_missing,sources,expected_rows_10y,coverage_10y,short_history,has_quality_issue,company,sector
0,A,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Agilent Technologies,Health Care
1,AAPL,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Apple Inc.,Information Technology
2,ABBV,2637,2016-01-04,2026-06-30,0,1,0,yahoo,2738,0.9631,False,True,AbbVie,Health Care
3,ABNB,1393,2020-12-10,2026-06-30,0,0,0,yahoo,2738,0.5088,True,False,Airbnb,Consumer Discretionary
4,ABT,2637,2016-01-04,2026-06-30,0,1,0,yahoo,2738,0.9631,False,True,Abbott Laboratories,Health Care


,Ticker,company,sector,fail_stage,fail_reason,n_rows_raw,n_rows_final
0,HONA,Honeywell Aerospace,Industrials,collection,yahoo: rows=11 | chart: None,0,0


In [4]:
final_df.to_parquet(PROCESSED_DIR / 'final_df.parquet', index=False)
coverage_df.to_csv(PROCESSED_DIR / 'coverage_df.csv', index=False)
final_missing_df.to_csv(PROCESSED_DIR / 'final_missing_df.csv', index=False)
print(f'processed 저장 완료: {PROCESSED_DIR}')

processed 저장 완료: C:\workspaces\lab_middle_project\data\processed


In [5]:
print(
    "수집 성공 종목:",
    final_df_raw["Ticker"].nunique(),
)
print(
    "수집 실패 종목:",
    collection_missing_df["ticker"].nunique(),
)
print(
    "전체 대상 종목:",
    sp500_universe["ticker"].nunique(),
)

수집 성공 종목: 502
수집 실패 종목: 1
전체 대상 종목: 503


# ML팀 전달 범위

`final_df`는 01 노트북에서 수집한 `data/raw/final_df_raw.parquet`를 `build_ml_dataset()`으로 정제한 결과다. 분석 기간 제한, 날짜·티커 형식 통일, 유효하지 않은 종가 제거, `(Ticker, Date)` 중복 제거, OHLC 보정, 거래량 결측 처리와 정렬을 수행한 뒤 `data/processed/final_df.parquet`로 저장한다. 지표는 아직 추가되지 않은 전체 일별 OHLCV 데이터다. 기간별 `final_20_df`, `final_60_df`, `final_120_df`, `final_252_df`는 만들거나 저장하지 않는다.

피처 생성은 `src/feature.py`를 사용하며, ML팀이 분석 목적에 맞게 `windows`를 명시한다. 생성 대상은 `beta`, `volatility`, `return`, `rsi` 네 지표다. beta 계산에는 `data/raw/sp500_beta_df.parquet`가 추가로 필요하다.

`final_df`에는 분석 기간 중 한 번이라도 S&P 500 구성종목이었던 티커의 가격이 들어 있지만, 각 행을 해당 날짜의 실제 구성종목 여부로 미리 필터링하지는 않았다. 따라서 과거 리밸런싱 시점에 아직 편입되지 않았거나 이미 편출된 종목을 선택하는 오류를 막으려면 `src/get_tickers.py`의 `filter_by_membership()`과 이 함수가 읽는 `data/raw/cache/sp500_membership_history.csv`도 함께 전달해야 한다. `get_historical_sp500_universe()`는 전체 가격 수집 대상을 정하는 함수이고, 시점별 리밸런싱 후보 제한에는 `filter_by_membership()`을 사용한다.

리밸런싱 주기는 피처 window와 별개다. 전체 일별 데이터를 유지한 상태에서 ML팀이 백테스트 기준에 따라 리밸런싱 날짜, 종목 재선정, 목표 비중, 거래비용을 결정한다. 각 리밸런싱 날짜의 후보군에는 반드시 `filter_by_membership()`을 적용한다. 일정 간격의 행만 미리 추출한 데이터프레임은 실제 리밸런싱 결과가 아니므로 이 단계에서 생성하지 않는다.

사용 예시:

```python
from src.feature import add_features

featured_df = add_features(
    prices=final_df,
    sp500_beta_df=sp500_beta_df,
    windows=(20, 60, 120, 252),  # ML팀이 후보 기간을 직접 지정
)
```